# Task 1 — Exploratory Data Analysis

**Dataset:** Synthetic e-commerce orders (Jan 2023 – Jun 2025)

**Goal:** Understand the structure, quality, and basic distributions of the data before any deeper analysis is done in Task 2.

## 1. Load Data

In [ ]:
import pandas as pd
import numpy as np

# ---------------------------------------------------------------
# Self-contained data generation
# This block is identical across all three notebooks so each one
# can run independently (no shared kernel state required).
# Seeded for reproducibility.
# ---------------------------------------------------------------
np.random.seed(42)

n = 1200
categories = ['Electronics','Clothing','Home & Kitchen','Books','Beauty','Sports','Toys','Grocery']
payment_methods = ['Credit Card','Debit Card','Cash on Delivery','Digital Wallet']
regions = ['North','South','East','West','Central']
statuses = ['Delivered','Cancelled','Returned','Pending']
status_weights = [0.62, 0.16, 0.14, 0.08]

start_date = pd.Timestamp('2023-01-01')
end_date = pd.Timestamp('2025-06-30')
date_range_days = (end_date - start_date).days

df = pd.DataFrame({
    'order_id': [f'ORD{100000+i}' for i in range(n)],
    'customer_id': np.random.randint(1000, 1500, size=n),
    'order_date': [start_date + pd.Timedelta(days=int(x)) for x in np.random.randint(0, date_range_days, size=n)],
    'product_category': np.random.choice(categories, size=n),
    'quantity': np.random.randint(1, 6, size=n),
    'unit_price': np.round(np.random.gamma(3, 25, size=n), 2),
    'discount_percent': np.random.choice([0,5,10,15,20,25], size=n, p=[0.35,0.2,0.2,0.15,0.07,0.03]),
    'payment_method': np.random.choice(payment_methods, size=n),
    'shipping_cost': np.round(np.random.uniform(2,15,size=n),2),
    'delivery_days': np.random.randint(1,15,size=n),
    'customer_rating': np.random.choice([1,2,3,4,5], size=n, p=[0.05,0.08,0.17,0.35,0.35]),
    'order_status': np.random.choice(statuses, size=n, p=status_weights),
    'region': np.random.choice(regions, size=n),
})

df['total_amount'] = np.round(
    df['quantity'] * df['unit_price'] * (1 - df['discount_percent']/100) + df['shipping_cost'], 2
)

# Inject realistic data-quality issues so the pipeline has something genuine to catch
missing_idx = np.random.choice(df.index, size=25, replace=False)
df.loc[missing_idx, 'customer_rating'] = np.nan
df = pd.concat([df, df.sample(8, random_state=1)], ignore_index=True)

print(f"Dataset loaded: {df.shape[0]} rows x {df.shape[1]} columns")
df.head()

## 2. Structure & Data Types

In [ ]:
df.info()

In [ ]:
df.describe(include='all').T

## 3. Data Quality Checks

Before drawing any conclusions we check for missing values and duplicate rows. Skipping this step is the single most common reason an EDA's later findings turn out to be wrong.

In [ ]:
missing = df.isnull().sum()
missing = missing[missing > 0]
print('Missing values by column:')
print(missing if len(missing) else 'None found')
print(f"\nMissing as % of rows: {(missing.sum()/len(df))*100:.2f}%" if len(missing) else '')

In [ ]:
dupes = df.duplicated().sum()
print(f'Duplicate rows: {dupes} ({dupes/len(df)*100:.2f}% of dataset)')

In [ ]:
# Clean a working copy: drop duplicates, impute missing rating with the median
df_clean = df.drop_duplicates().copy()
df_clean['customer_rating'] = df_clean['customer_rating'].fillna(df_clean['customer_rating'].median())
print(f'Rows before cleaning: {len(df)}')
print(f'Rows after cleaning:  {len(df_clean)}')
assert df_clean.isnull().sum().sum() == 0, 'Unexpected missing values remain'
assert df_clean.duplicated().sum() == 0, 'Unexpected duplicates remain'
print('✓ Data quality checks passed')

In [ ]:
df_clean.to_csv('ecommerce_orders_clean.csv', index=False)
print('Saved cleaned dataset to ecommerce_orders_clean.csv')

## 4. Univariate Distributions

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

num_cols = ['quantity','unit_price','discount_percent','shipping_cost',
            'delivery_days','customer_rating','total_amount']

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    sns.histplot(df_clean[col], kde=True, ax=axes[i], color='steelblue')
    axes[i].set_title(col)
for j in range(len(num_cols), len(axes)):
    fig.delaxes(axes[j])
plt.tight_layout()
plt.show()

## 5. Categorical Breakdown

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
sns.countplot(data=df_clean, x='product_category', order=df_clean['product_category'].value_counts().index, ax=axes[0,0])
axes[0,0].set_title('Orders by Category'); axes[0,0].tick_params(axis='x', rotation=45)

sns.countplot(data=df_clean, x='order_status', order=df_clean['order_status'].value_counts().index, ax=axes[0,1])
axes[0,1].set_title('Orders by Status')

sns.countplot(data=df_clean, x='payment_method', order=df_clean['payment_method'].value_counts().index, ax=axes[1,0])
axes[1,0].set_title('Orders by Payment Method'); axes[1,0].tick_params(axis='x', rotation=30)

sns.countplot(data=df_clean, x='region', order=df_clean['region'].value_counts().index, ax=axes[1,1])
axes[1,1].set_title('Orders by Region')
plt.tight_layout()
plt.show()

In [ ]:
status_pct = df_clean['order_status'].value_counts(normalize=True) * 100
print('Order status breakdown (%):')
print(status_pct.round(2))

cancel_return_rate = status_pct.get('Cancelled', 0) + status_pct.get('Returned', 0)
print(f"\nCombined cancellation + return rate: {cancel_return_rate:.2f}%")

## 6. Summary

- Dataset loaded with data-quality issues intentionally present, then explicitly detected and resolved (rather than assumed away).
- Missing values and duplicate rows are quantified, not just visually implied.
- A cleaned copy (`ecommerce_orders_clean.csv`) is saved for Task 2 and Task 3 to load independently.
- Combined cancellation/return rate is computed directly from the data — see printed output above for the exact figure, rather than a rounded guess.